# 04 — Run the Trained Model Live

Load the current trained MOUSE model from the Hub and evaluate it on Procedural Frozen Lake.

Both training notebooks push to the same `MODEL_ID`. This notebook pulls whatever checkpoint is currently on the Hub, so if you run offline training and push, it evaluates that model; if you run online training and push afterward, it evaluates the online-trained model instead.

This notebook covers two things:
1. **Success rate** — run the model for `EVAL_EPISODES` episodes and report how often it reaches the goal.
2. **Replay animation** — capture a short episode and play it back as an inline animation.

In [6]:
import os

import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

from mouse_envs import EnvConfig, make_env
from mouse_core import load_model

# FrozenLake renders via pygame; run headless in environments without a display.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

MODEL_ID = "mouse-example-model"
TOTAL_STEPS = 200
NUM_ENVS = 100
NUM_VIDEOS = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Environment

`NUM_ENVS` independent env instances run in parallel. Each one is specified by its own `EnvConfig`, with `reset_seed=i` for its reset stream and `map_seed=i` for its deterministic Procedural Frozen Lake map. Each env runs one infinite task (`episodes_per_task=0`) with deterministic actions (`is_slippery=False`) and 50-step episode truncation. Only the first `NUM_VIDEOS` envs use `render_mode="rgb_array"`, so evaluation can cover more envs without embedding a video for every one.

In [7]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_{i}",
        reset_seed=i,
        episodes_per_task=0,
        kwargs={
            "is_slippery": False,
            "min_width": 4, "max_width": 8,
            "min_height": 4, "max_height": 8,
            "step_penalty": -0.01,
            **({"render_mode": "rgb_array"} if i < NUM_VIDEOS else {}),
            "max_episode_steps": 50,
            "map_seed": i+1000000,
        },
    )
    for i in range(NUM_ENVS)
]

env = make_env(configs)
print("Environments:", env.names)

Environments: ('proc_frozenlake_0', 'proc_frozenlake_1', 'proc_frozenlake_2', 'proc_frozenlake_3', 'proc_frozenlake_4', 'proc_frozenlake_5', 'proc_frozenlake_6', 'proc_frozenlake_7', 'proc_frozenlake_8', 'proc_frozenlake_9', 'proc_frozenlake_10', 'proc_frozenlake_11', 'proc_frozenlake_12', 'proc_frozenlake_13', 'proc_frozenlake_14', 'proc_frozenlake_15', 'proc_frozenlake_16', 'proc_frozenlake_17', 'proc_frozenlake_18', 'proc_frozenlake_19', 'proc_frozenlake_20', 'proc_frozenlake_21', 'proc_frozenlake_22', 'proc_frozenlake_23', 'proc_frozenlake_24', 'proc_frozenlake_25', 'proc_frozenlake_26', 'proc_frozenlake_27', 'proc_frozenlake_28', 'proc_frozenlake_29', 'proc_frozenlake_30', 'proc_frozenlake_31', 'proc_frozenlake_32', 'proc_frozenlake_33', 'proc_frozenlake_34', 'proc_frozenlake_35', 'proc_frozenlake_36', 'proc_frozenlake_37', 'proc_frozenlake_38', 'proc_frozenlake_39', 'proc_frozenlake_40', 'proc_frozenlake_41', 'proc_frozenlake_42', 'proc_frozenlake_43', 'proc_frozenlake_44', 'proc

## Load model

`load_model` downloads the current checkpoint from the shared Hub repo (if not already cached) and reconstructs the full `Model` object with its saved weights. You don't need to re-specify the architecture — it's stored alongside the weights.

In [ ]:
model = load_model(MODEL_ID, map_location="cpu").eval().to(device)

: 

## Incremental inference with KV-cache

At inference time you don't need to re-process the full history on every step. Backbones that support it (`Qwen3Backbone`, `LlamaBackbone`) cache the key/value states from previous tokens so each new step only processes the single new token.

Pass `use_cache=True` and thread the cache object across steps:

```python
cache = None

with torch.no_grad():
    predictions, _, cache = model([[step_dict]], cache=cache, use_cache=True)
    action = model.get_action(predictions, temperature=0.0, num_actions=env.action_space.spaces[i].n)
```

`model.get_action` reads the action head's output from the last step in `predictions`. Use `temperature=0.0` for greedy argmax; increase it above 0 to sample stochastically.

**When to reset the cache:** reset `cache = None` only at **task** boundaries (done code 3 or 4). This example uses `episodes_per_task=0`, so the cache normally carries for the whole run, including across episode boundaries.

In [ ]:
caches = [None] * len(env.names)  # one KV-cache per env instance
video_names = env.names[:NUM_VIDEOS]
video_envs = env._env_instances[:NUM_VIDEOS]
frames_per_env = [[] for _ in video_names]
inputs = None
outputs = None
env.tracker.clear()
for t in range(TOTAL_STEPS):
    if outputs is None:
        inputs = env.sample_random_inputs()
    else:
        with torch.no_grad():
            new_inputs = []
            new_caches = []
            for i, (inp, out) in enumerate(zip(inputs, outputs)):
                pred, _, new_cache = model([[{**inp, **out}]], cache=caches[i], use_cache=True)
                action = model.get_action(pred, temperature=0.0, num_actions=env.action_space.spaces[i].n)
                new_inputs.append({"action": action.squeeze().cpu()})
                new_caches.append(new_cache)
            inputs = new_inputs
            caches = new_caches
    outputs = env.step(inputs)
    for i, video_env in enumerate(video_envs):
        frames = video_env.render()
        if frames:
            frames_per_env[i].append(frames[-1])

env.close()

for name, rewards, lengths in zip(env.names, env.tracker.episode_cum_rewards, env.tracker.episode_lengths):
    n = len(rewards)
    mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
    mean_l = torch.tensor(lengths).mean().item() if n else float("nan")
    print(f"{name}: {n} episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}")

/home/user/Repos/mouse-core/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/home/user/Repos/mouse-core/.venv/lib/python3.12/site-packages/gymnasium/envs/toy_text/frozen_lake.py:352: UserWarning: WARN: You are calling render method without specifying any render mode. You can specify the render_mode at initialization, e.g. gym.make("Procedural-FrozenLake-v1", render_mode="rgb_array")
  gym.logger.warn(


proc_frozenlake_0: 100 episodes | mean reward -0.010 | mean length 1.0
proc_frozenlake_1: 3 episodes | mean reward -0.500 | mean length 50.0
proc_frozenlake_2: 66 episodes | mean reward -0.020 | mean length 2.0
proc_frozenlake_3: 3 episodes | mean reward -0.500 | mean length 50.0
proc_frozenlake_4: 3 episodes | mean reward -0.500 | mean length 50.0
proc_frozenlake_5: 3 episodes | mean reward -0.500 | mean length 50.0
proc_frozenlake_6: 3 episodes | mean reward -0.500 | mean length 50.0
proc_frozenlake_7: 66 episodes | mean reward -0.020 | mean length 2.0
proc_frozenlake_8: 3 episodes | mean reward -0.500 | mean length 50.0
proc_frozenlake_9: 3 episodes | mean reward -0.500 | mean length 50.0
proc_frozenlake_10: 3 episodes | mean reward -0.500 | mean length 50.0
proc_frozenlake_11: 3 episodes | mean reward -0.500 | mean length 50.0
proc_frozenlake_12: 49 episodes | mean reward -0.030 | mean length 3.0
proc_frozenlake_13: 33 episodes | mean reward 0.950 | mean length 5.0
proc_frozenlake_

## Replay MP4s

Frames were collected from the first `NUM_VIDEOS` env instances during the evaluation run above. Each captured env instance is displayed as its own embedded MP4 so the notebook output stays self-contained without creating separate media files to version.

In [ ]:
def display_env_replay(name, frames, *, fps=5):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.axis("off")
    ax.set_title(name, fontsize=10)
    img = ax.imshow(frames[0])

    def update(t):
        img.set_data(frames[t])
        return (img,)

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=1000 / fps,
        blit=True,
    )
    plt.close(fig)
    display(HTML(ani.to_html5_video()))


for name, frames in zip(video_names, frames_per_env):
    display_env_replay(name, frames)